# Missingness Indicators and Feature Retention Experiments

## Project
Machine Learning-Based Air Quality Classification System

## Objective
Investigate whether explicitly representing missing pollutant measurements improves AQI classification and whether retaining additional pollutant features improves model performance.

## Experiments
1. Median imputation vs. median imputation with missingness indicators.
2. Retaining 10, 11, and 12 pollutant features.

## Evaluation
- Accuracy
- Macro F1
- Weighted F1
- Class-wise precision and recall

A chronological validation strategy will be used for model selection.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42

In [2]:
data_path = Path("../data/raw/city_hour.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset shape: (707875, 16)

Columns:
['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']


,City,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01 01:00:00,NaN,NaN,1.00,40.01,36.37,NaN,1.00,122.07,NaN,0.0,0.0,0.0,NaN,NaN
1,Ahmedabad,2015-01-01 02:00:00,NaN,NaN,0.02,27.75,19.73,NaN,0.02,85.90,NaN,0.0,0.0,0.0,NaN,NaN
2,Ahmedabad,2015-01-01 03:00:00,NaN,NaN,0.08,19.32,11.08,NaN,0.08,52.83,NaN,0.0,0.0,0.0,NaN,NaN
3,Ahmedabad,2015-01-01 04:00:00,NaN,NaN,0.30,16.45,9.20,NaN,0.30,39.53,153.58,0.0,0.0,0.0,NaN,NaN
4,Ahmedabad,2015-01-01 05:00:00,NaN,NaN,0.12,14.90,7.85,NaN,0.12,32.63,NaN,0.0,0.0,0.0,NaN,NaN


In [3]:
# Define the target and pollutant features

target = "AQI_Bucket"

features_10 = [
    "PM2.5", "PM10", "NO", "NO2", "NOx",
    "NH3", "CO", "SO2", "O3", "Benzene"
]

features_11 = features_10 + ["Toluene"]

features_12 = features_11 + ["Xylene"]

# Keep only observations with known target labels
model_df = df.loc[df[target].notna()].copy()

print("Labelled observations:", len(model_df))
print("10-feature configuration:", len(features_10))
print("11-feature configuration:", len(features_11))
print("12-feature configuration:", len(features_12))

Labelled observations: 578795
10-feature configuration: 10
11-feature configuration: 11
12-feature configuration: 12


In [4]:
# Convert Datetime to datetime format
model_df["Datetime"] = pd.to_datetime(model_df["Datetime"])

# Get unique timestamps in chronological order
timestamps = np.sort(model_df["Datetime"].unique())

# First split: 80% development period, 20% later test period
test_cutoff = timestamps[int(len(timestamps) * 0.80)]

development_df = model_df[
    model_df["Datetime"] < test_cutoff
].copy()

test_df = model_df[
    model_df["Datetime"] >= test_cutoff
].copy()

# Second split: 80% fitting period, 20% validation period
development_timestamps = np.sort(
    development_df["Datetime"].unique()
)

validation_cutoff = development_timestamps[
    int(len(development_timestamps) * 0.80)
]

train_df = development_df[
    development_df["Datetime"] < validation_cutoff
].copy()

validation_df = development_df[
    development_df["Datetime"] >= validation_cutoff
].copy()

print("TRAINING SET")
print("Shape:", train_df.shape)
print("Period:", train_df["Datetime"].min(), "to", train_df["Datetime"].max())

print("\nVALIDATION SET")
print("Shape:", validation_df.shape)
print("Period:", validation_df["Datetime"].min(), "to", validation_df["Datetime"].max())

print("\nTEST SET")
print("Shape:", test_df.shape)
print("Period:", test_df["Datetime"].min(), "to", test_df["Datetime"].max())

TRAINING SET
Shape: (242225, 16)
Period: 2015-01-01 16:00:00 to 2018-07-09 07:00:00

VALIDATION SET
Shape: (130370, 16)
Period: 2018-07-09 08:00:00 to 2019-05-26 12:00:00

TEST SET
Shape: (206200, 16)
Period: 2019-05-26 13:00:00 to 2020-07-01 00:00:00


In [5]:
# Separate features and target

X_train = train_df[features_10].copy()
y_train = train_df[target].copy()

X_val = validation_df[features_10].copy()
y_val = validation_df[target].copy()

print("Training features:", X_train.shape)
print("Training target:", y_train.shape)

print("\nValidation features:", X_val.shape)
print("Validation target:", y_val.shape)

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nValidation class distribution:")
print(y_val.value_counts().sort_index())

Training features: (242225, 10)
Training target: (242225,)

Validation features: (130370, 10)
Validation target: (130370,)

Training class distribution:
AQI_Bucket
Good            10434
Moderate        83876
Poor            34947
Satisfactory    63431
Severe          15371
Very Poor       34166
Name: count, dtype: int64

Validation class distribution:
AQI_Bucket
Good             4378
Moderate        49578
Poor            16502
Satisfactory    38766
Severe           7820
Very Poor       13326
Name: count, dtype: int64


In [6]:
# Fit median imputer on training data
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

print("Training shape:", X_train_imputed.shape)
print("Validation shape:", X_val_imputed.shape)

Training shape: (242225, 10)
Validation shape: (130370, 10)


In [7]:
baseline_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

baseline_model.fit(X_train_imputed, y_train)

y_val_pred_baseline = baseline_model.predict(X_val_imputed)

In [8]:
print("Accuracy:", accuracy_score(y_val, y_val_pred_baseline))
print("Macro F1:", f1_score(y_val, y_val_pred_baseline, average="macro"))
print("Weighted F1:", f1_score(y_val, y_val_pred_baseline, average="weighted"))

Accuracy: 0.5930275370100483
Macro F1: 0.443902273181306
Weighted F1: 0.5695217387938798


In [9]:
# Create missingness indicators from original data
X_train_missing = X_train.isna().astype(int)
X_val_missing = X_val.isna().astype(int)

# Rename indicator columns
X_train_missing.columns = [
    col + "_missing" for col in X_train_missing.columns
]

X_val_missing.columns = [
    col + "_missing" for col in X_val_missing.columns
]

print("Training indicators:", X_train_missing.shape)
print("Validation indicators:", X_val_missing.shape)

print("\nTraining missingness counts:")
print(X_train_missing.sum().sort_values(ascending=False))

Training indicators: (242225, 10)
Validation indicators: (130370, 10)

Training missingness counts:
PM10_missing       118855
NH3_missing         99656
Benzene_missing     46894
NOx_missing         32502
CO_missing          17992
O3_missing          15986
SO2_missing         13916
PM2.5_missing       12098
NO_missing           9515
NO2_missing          8855
dtype: int64


In [10]:
# Convert imputed arrays back into DataFrames
X_train_imputed_df = pd.DataFrame(
    X_train_imputed,
    columns=features_10,
    index=X_train.index
)

X_val_imputed_df = pd.DataFrame(
    X_val_imputed,
    columns=features_10,
    index=X_val.index
)

# Combine pollutant values and missingness indicators
X_train_combined = pd.concat(
    [X_train_imputed_df, X_train_missing], axis=1
)

X_val_combined = pd.concat(
    [X_val_imputed_df, X_val_missing], axis=1
)

print("Combined training shape:", X_train_combined.shape)
print("Combined validation shape:", X_val_combined.shape)

Combined training shape: (242225, 20)
Combined validation shape: (130370, 20)


In [11]:
indicator_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

indicator_model.fit(X_train_combined, y_train)

y_val_pred_indicator = indicator_model.predict(X_val_combined)

In [12]:
print("Accuracy:", accuracy_score(y_val, y_val_pred_indicator))
print("Macro F1:", f1_score(y_val, y_val_pred_indicator, average="macro"))
print("Weighted F1:", f1_score(y_val, y_val_pred_indicator, average="weighted"))

Accuracy: 0.5930275370100483
Macro F1: 0.443902273181306
Weighted F1: 0.5695217387938798


In [13]:
# Compare predictions
print(
    "Identical predictions:",
    np.array_equal(y_val_pred_baseline, y_val_pred_indicator)
)

# Inspect feature importances
importance = pd.Series(
    indicator_model.feature_importances_,
    index=X_train_combined.columns
).sort_values(ascending=False)

print("\nFeature importances:")
print(importance)

Identical predictions: True

Feature importances:
PM2.5              0.743697
CO                 0.178671
O3                 0.034552
PM10               0.022802
NH3                0.012449
NOx                0.005933
NO                 0.001688
NO2                0.000164
NH3_missing        0.000045
SO2                0.000000
PM2.5_missing      0.000000
Benzene            0.000000
PM10_missing       0.000000
NO_missing         0.000000
NO2_missing        0.000000
NOx_missing        0.000000
CO_missing         0.000000
SO2_missing        0.000000
O3_missing         0.000000
Benzene_missing    0.000000
dtype: float64


In [14]:
np.array_equal(y_val_pred_baseline, y_val_pred_indicator)

True

In [15]:
# Check missingness of the additional pollutants
for feature in ["Toluene", "Xylene"]:
    print(
        f"{feature}: "
        f"{train_df[feature].isna().mean():.2%} missing in training, "
        f"{validation_df[feature].isna().mean():.2%} missing in validation"
    )

Toluene: 21.51% missing in training, 28.40% missing in validation
Xylene: 65.54% missing in training, 65.55% missing in validation


In [16]:
feature_sets = {
    "10_features": features_10,
    "11_features": features_11,
    "12_features": features_12
}

results = []

for name, features in feature_sets.items():

    # Select features
    X_tr = train_df[features]
    X_v = validation_df[features]

    # Impute missing values
    imputer = SimpleImputer(strategy="median")

    X_tr_imputed = imputer.fit_transform(X_tr)
    X_v_imputed = imputer.transform(X_v)

    # Train model
    model = DecisionTreeClassifier(
        max_depth=5,
        random_state=RANDOM_STATE
    )

    model.fit(X_tr_imputed, y_train)

    # Predict
    predictions = model.predict(X_v_imputed)

    # Evaluate
    results.append({
        "Feature Set": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Macro F1": f1_score(y_val, predictions, average="macro"),
        "Weighted F1": f1_score(y_val, predictions, average="weighted")
    })

results_df = pd.DataFrame(results)
print(results_df.round(4))

   Feature Set  Accuracy  Macro F1  Weighted F1
0  10_features     0.593    0.4439       0.5695
1  11_features     0.593    0.4439       0.5695
2  12_features     0.593    0.4439       0.5695


In [17]:
# Inspect feature importance for each feature configuration

for name, features in feature_sets.items():

    X_tr = train_df[features]

    imputer = SimpleImputer(strategy="median")
    X_tr_imputed = imputer.fit_transform(X_tr)

    model = DecisionTreeClassifier(
        max_depth=5,
        random_state=RANDOM_STATE
    )

    model.fit(X_tr_imputed, y_train)

    importance = pd.Series(
        model.feature_importances_,
        index=features
    ).sort_values(ascending=False)

    print(f"\n{name}")
    print(importance)


10_features
PM2.5      0.743697
CO         0.178671
O3         0.034552
PM10       0.022802
NH3        0.012493
NOx        0.005933
NO         0.001688
NO2        0.000164
SO2        0.000000
Benzene    0.000000
dtype: float64

11_features
PM2.5      0.743697
CO         0.178671
O3         0.034552
PM10       0.022802
NH3        0.012493
NOx        0.005933
NO         0.001688
NO2        0.000164
SO2        0.000000
Benzene    0.000000
Toluene    0.000000
dtype: float64

12_features
PM2.5      0.743697
CO         0.178671
O3         0.034552
PM10       0.022802
NH3        0.012493
NOx        0.005933
NO         0.001688
NO2        0.000164
SO2        0.000000
Benzene    0.000000
Toluene    0.000000
Xylene     0.000000
dtype: float64
